# Procedural transformations on time-series windows

We transform windows cut from a real dataset, [ETT](https://github.com/zhouhaoyi/ETDataset), with the three procedural transformations of `kcai-data-sampling-timeseries`:

| Transformation | Parameters | Support | Randomness |
|---|---|---|---|
| `low_pass` | `cutoff`, cycles per step in `(0, 0.5]` | global: each output step depends on the whole window | none |
| `add_sinusoid` | `frequency` (cycles per step), `amplitude`, `phase` | pointwise | none |
| `jitter` | `sigma`, in the units of the signal | pointwise | `seed` |

The sample is a **window**: `W` consecutive steps of one series over `C` channels. A transformation receives a batch `(B, W, C)` of float windows, with axes `(time, channel)` and no value range, and returns a batch of the same shape. The `timeseries_parquet` loader cuts the windows, and its `channels` decides which channels a window holds: the transformation transforms all of them. Each window carries its **key**: `series_id`, `start_window`, `end_window` and, optionally, `t0_series`. Every relation between two windows (same series, overlap, order and lag, simultaneity) is computed from it. Section 7 sketches how a transformation could use these relations to keep its outputs consistent: that part is **not coded** in the package, it is notebook code.

Frequencies are in **cycles per step**: a transformation sees the window array only, not the sampling rate. ETT is hourly, so one step is one hour: the daily cycle is `1/24 ≈ 0.042` cycles per step. The plots show frequencies in cycles per day.

**Data.** ETT (Electricity Transformer Temperature; Zhou et al., Informer, AAAI 2021): two transformers, `ETTh1` and `ETTh2`, recorded every hour over the same two years, 17,420 steps each, with 7 channels: six loads (`HUFL`, `HULL`, `MUFL`, `MULL`, `LUFL`, `LULL`: high, middle and low, useful and useless load) and the oil temperature `OT`. It is downloaded by `scripts/fetch_ett_sample.py` (the setup cell runs it if needed). ETT is licensed CC BY-ND 4.0: neither the data nor the outputs of this notebook are to be committed or shared, so the notebook is committed without its outputs.

To run it from the repository root: `uv run --with jupyterlab --with matplotlib jupyter lab examples/notebooks/timeseries-transformations.ipynb`. `matplotlib` is used for the plots only.

## Setup

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import yaml

%config InlineBackend.figure_format = 'png'
pd.set_option("display.max_colwidth", None)

# Work from the repository root: every path in the config is relative to it.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(REPO)

from kcai_data_sampling_job.cli import run
from kcai_data_sampling_job.dataloaders.api.timeseries import TimeSeriesLoaderConfig, TimeSeriesParquetDataLoader
from kcai_data_sampling_timeseries.api.transformations import AddSinusoid, Jitter, LowPass

with open("examples/config/ett-procedural.yaml") as f:
    CFG = yaml.safe_load(f)
LOADER = CFG["dataloaders"]["loaders"][0]
if not Path(LOADER["path"]).exists():
    subprocess.run([sys.executable, "scripts/fetch_ett_sample.py"], check=True)
print(yaml.safe_dump(CFG, sort_keys=False))

## 1. The series

One row per transformer: an id, the values (`list<list<float>>`, 17,420 hourly steps of 7 channels), and two source columns, `start` (the timestamp of the first step) and `freq`. We look at two channels of `ETTh1`: the high useful load `HUFL` (channel 0) and the oil temperature `OT` (channel 6). The oil temperature follows the seasons; both channels carry a daily cycle.

In [ ]:
table = pq.read_table(LOADER["path"])
print(table.schema)
print(table.select(["series_id", "start", "freq"]).to_pandas())

CHANNEL_NAMES = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]
series = np.asarray(table.column("values")[0].as_py())  # ETTh1, (T, C)
dates = pd.date_range(table.column("start")[0].as_py(), periods=len(series), freq="h")
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
for ax, c in zip(axes, [0, 6]):
    ax.plot(dates, series[:, c], lw=0.4)
    ax.set_ylabel(CHANNEL_NAMES[c])
fig.suptitle(f"{table.column('series_id')[0]}: {len(series):,} hourly steps")
plt.show()

## 2. Windows and their key

The example config cuts **consecutive** windows of `W = 168` steps (one week) every `stride = 84` steps (half a week): 206 windows per transformer, each sharing 84 steps with the next one.

The sample has a clock: `start` gives the timestamp of each series' first step. The loader reads a clock from an integer column of steps, `t0_column`: we compute it from `start`, in hours from the earliest series, and write it in a `t0` column of a copy of the table. The loader carries it as `t0_series` on every window. The two transformers start at the same hour, so both get `t0 = 0`: their windows of the same index are simultaneous.

`windows_of` reads every window of a selection, as a job would. The key table holds one row per window: the key, the id, and the source columns.

In [ ]:
def windows_of(windows, **loader):
    # Every window of the selection: the (N, W, C) stack and its key table.
    config = TimeSeriesLoaderConfig.model_validate({**LOADER, **loader, "windows": windows})
    (selection,) = TimeSeriesParquetDataLoader(config.name, config).get_selections()
    selection.bootstrap(None)
    batches = list(selection)
    keys = pd.concat([b.columns.to_pandas().assign(id=b.ids) for b in batches], ignore_index=True)
    return np.concatenate([b.data for b in batches]), keys.drop(columns=["start", "freq"])


starts = pd.to_datetime(table.column("start").to_pylist())
T0 = dict(zip(table.column("series_id").to_pylist(), ((starts - starts.min()) // pd.Timedelta(hours=1)).tolist()))
print("t0 (hours from the earliest series):", T0)
T0_PATH = Path(LOADER["path"]).with_name("series_t0.parquet")
pq.write_table(table.append_column("t0", pa.array(list(T0.values()), pa.int64())), T0_PATH)

data, keys = windows_of(LOADER["windows"], path=str(T0_PATH), t0_column="t0", channels=None)
PER_SERIES = int((keys.series_id == "ETTh1").sum())
print(f"{len(keys)} windows ({PER_SERIES} per transformer), data {data.shape} {data.dtype}")
keys.iloc[[0, 1, 2, 3, PER_SERIES, PER_SERIES + 1]]

Relations between two windows are computed from the key, never stored:

- **same series**: equal `series_id`;
- **overlap**, same series: `max(0, min(e₁, e₂) − max(s₁, s₂) + 1)` shared steps;
- **order and lag**, same series: when `s₂ > e₁`, window 1 precedes window 2 by `s₂ − e₁ − 1` steps;
- **simultaneity**, two series: the clock intervals `[t0_series + s, t0_series + e]` intersect.

Windows 0 and 2 of `ETTh1` touch (lag 0), windows 0 and 3 are 84 hours apart. The first window of `ETTh2` covers the same hours as window 0 of `ETTh1`, half of window 1, and none of window 2.

In [ ]:
def shared_steps(s1, e1, s2, e2):
    return max(0, min(e1, e2) - max(s1, s2) + 1)


def relation(a, b):
    if a.series_id == b.series_id:
        shared = shared_steps(a.start_window, a.end_window, b.start_window, b.end_window)
        if shared:
            return f"same series, overlap of {shared} steps"
        return f"same series, lag of {b.start_window - a.end_window - 1} steps"
    clock = shared_steps(
        a.t0_series + a.start_window, a.t0_series + a.end_window, b.t0_series + b.start_window, b.t0_series + b.end_window
    )
    return f"two series, simultaneous on {clock} steps" if clock else "two series, not simultaneous"


w = dict(enumerate(keys.itertuples()))
for i, j in [(0, 1), (0, 2), (0, 3), (0, PER_SERIES), (1, PER_SERIES), (2, PER_SERIES)]:
    print(f"window {i:>3} ({w[i].series_id}) vs window {j:>3} ({w[j].series_id}): {relation(w[i], w[j])}")

**Independent** windows are drawn: `n_windows` positions per series, seeded by `seed` and the series id. With `non_overlap: true`, no two windows of a series share a step. The same seed gives the same windows. An id depends only on `(series, start, end)`, so a window gets the same id whichever mode cut it.

In [ ]:
independent = {"mode": "independent", "window": 168, "n_windows": 3, "non_overlap": True, "seed": 0}
_, drawn = windows_of(independent, path=str(T0_PATH), t0_column="t0")
_, again = windows_of(independent, path=str(T0_PATH), t0_column="t0")
_, other = windows_of({**independent, "seed": 1}, path=str(T0_PATH), t0_column="t0")
print("same seed, same windows:", drawn.equals(again))
print("other seed, other windows:", not drawn[["start_window"]].equals(other[["start_window"]]))
drawn

## 3. Selecting a channel, applying a transformation

A transformation transforms **every channel** of the windows it receives. Which channels a window holds is a choice of the **data selection**: the loader's `channels` keeps the listed ones. The example config keeps the oil temperature only (`channels: [6]`). Here we declare two selections over the same windows, one per channel:

| Selection | Loader | Window | Units |
|---|---|---|---|
| load | `channels: [0]` | `(168, 1)` | `HUFL`, as recorded |
| temperature | `channels: [6]` | `(168, 1)` | `OT`, °C |

Each transformation below is applied to each selection, with values suited to its channel:

| Transformation | On load | On temperature |
|---|---|---|
| `low_pass` | `cutoff = 0.06`: keeps the daily cycle | `cutoff = 0.03`: removes it |
| `add_sinusoid` | an 8-hour wave, `amplitude = 0.5` | a 6-hour wave, `amplitude = 1` °C |
| `jitter` | `sigma = 0.2` | `sigma = 0.3` °C |

We show the windows of `ETTh1`.

In [ ]:
CHANNEL_0 = {"name": "load", "unit": "HUFL", "channels": [0]}
CHANNEL_6 = {"name": "temperature", "unit": "°C", "channels": [6]}

data_l, keys_l = windows_of(LOADER["windows"], path=str(T0_PATH), t0_column="t0", channels=[0])
data_t, keys_t = windows_of(LOADER["windows"], path=str(T0_PATH), t0_column="t0", channels=[6])
print(f"load selection {data_l.shape}, temperature selection {data_t.shape}")
print("same windows, same ids:", keys_l.equals(keys) and keys_t.equals(keys))
print("each selection holds its channel:", np.array_equal(data_l[..., 0], data[..., 0]), np.array_equal(data_t[..., 0], data[..., 6]))

mask = (keys.series_id == "ETTh1").to_numpy()
xl, xt, ids = data_l[mask], data_t[mask], keys.id[mask].tolist()
steps = np.arange(xl.shape[1])
FLOOR = 1e-3


def transform(cls, xs, window_ids=None, **params):
    transformation = cls(params)
    return transformation.apply(xs, transformation.rngs(ids if window_ids is None else window_ids))


def spectrum(window):
    # Single-sided amplitude spectrum of one channel, in the channel's units, against frequency in cycles per day.
    amplitude = np.abs(np.fft.rfft(window)) / len(window)
    amplitude[1:] *= 2  # fold the negative frequencies in; 0 (the mean) is not folded
    if len(window) % 2 == 0:
        amplitude[-1] /= 2  # nor is the Nyquist bin
    return np.fft.rfftfreq(len(window), 1 / 24), amplitude


def show_transform(before, after, label, channel, window=0, mark=None):
    # One window of a one-channel selection, before (blue) and after (orange): the signal and its amplitude spectrum.
    fig, (ax_t, ax_f) = plt.subplots(1, 2, figsize=(13, 3), gridspec_kw={"width_ratios": [3, 2]})
    for xs_, lab in [(before, "before"), (after, f"after: {label}")]:
        ax_t.plot(steps, xs_[window, :, 0], lw=0.9, label=lab)
        f, a = spectrum(xs_[window, :, 0])
        ax_f.semilogy(f, np.maximum(a, FLOOR), lw=0.9, label=lab)
    if mark is not None:
        ax_f.axvline(mark * 24, color="grey", ls="--", lw=0.8)
    ax_t.set_ylabel(f"{channel['name']} ({channel['unit']})")
    ax_t.set_xlabel("hour in the window")
    ax_t.set_xticks(range(0, 169, 24))
    ax_t.legend(loc="upper right", fontsize=8)
    ax_f.set_ylim(FLOOR / 2, None)
    ax_f.set_ylabel(f"amplitude ({channel['unit']})")
    ax_f.set_xlabel("frequency (cycles per day)")
    fig.suptitle(
        f"{channel['name']} selection (channels: {channel['channels']}), window {window}: "
        "signal (left), single-sided amplitude spectrum, log scale (right)",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

## 4. `low_pass`: frequencies above `cutoff` are set to zero

It is an ideal FFT filter: every bin above `cutoff` is set to zero, and each output step depends on the whole window. A one-week window holds exactly 7 days, so the daily cycle sits on one FFT bin (7 cycles per window), and so do its harmonics (12 hours, 8 hours…): the filter keeps or removes each of them whole.

**On the temperature selection**, `cutoff = 0.03` cycles per hour, i.e. 0.72 cycles per day (dashed line): the daily cycle is removed, the slow change over the week stays.

In [ ]:
CUTOFF_L, CUTOFF_T = 0.06, 0.03
DAILY_BIN = 7  # 7 cycles in a 168-hour window
filtered_t = transform(LowPass, xt, cutoff=CUTOFF_T)
print(f"temperature, window 0, daily-cycle amplitude: {spectrum(xt[0, :, 0])[1][DAILY_BIN]:.2f} °C before, {spectrum(filtered_t[0, :, 0])[1][DAILY_BIN]:.1e} after")
show_transform(xt, filtered_t, "low_pass 0.03 (0.72 per day)", CHANNEL_6, mark=CUTOFF_T)

**On the load selection**, `cutoff = 0.06`, i.e. 1.44 cycles per day: the daily cycle stays, its harmonics (2 cycles per day and above) and the hour-to-hour fluctuations are removed.

In [ ]:
filtered_l = transform(LowPass, xl, cutoff=CUTOFF_L)
f, a_before = spectrum(xl[0, :, 0])
_, a_after = spectrum(filtered_l[0, :, 0])
print(f"load, window 0: daily cycle {a_before[DAILY_BIN]:.2f} -> {a_after[DAILY_BIN]:.2f}; 12-hour harmonic {a_before[2 * DAILY_BIN]:.2f} -> {a_after[2 * DAILY_BIN]:.1e}")
show_transform(xl, filtered_l, "low_pass 0.06 (1.44 per day)", CHANNEL_0, mark=CUTOFF_L)

The example config sweeps `cutoff` over 0.03, 0.06 and 0.09 cycles per hour, i.e. 0.72, 1.44 and 2.16 cycles per day. Each value sits on the other side of one harmonic: 0.03 removes the daily cycle, 0.06 keeps it, 0.09 also keeps the 12-hour harmonic. The band between two cutoffs holds more than the harmonic, though: real data has energy on every bin, and the bins next to the daily cycle are far from empty. The outputs differ by everything in that band:

In [ ]:
outs = {c: transform(LowPass, xt, cutoff=c) for c in (0.03, 0.06, 0.09)}
for a, b in [(0.03, 0.06), (0.06, 0.09)]:
    print(f"temperature: cutoffs {a} and {b} differ by up to {np.abs(outs[a] - outs[b]).max():.2f} °C")

f, amplitude = spectrum(xt[0, :, 0])
band = (f > 0.03 * 24) & (f <= 0.06 * 24)
print("window 0, bins between 0.03 and 0.06 (cycles per day):", dict(zip(f[band].round(2).tolist(), amplitude[band].round(2).tolist())))

**Overlapping windows are transformed separately.** Windows 0 and 1 share hours 84 to 167. A deterministic pointwise transformation would give the same values on them. `low_pass` reads the whole window, so the two windows give two different outputs for the same hours. The FFT also treats a window as periodic: it sees the jump from the last value back to the first one, and the filter rings at both edges of the window. On both selections, windows 0 and 1 disagree on their shared hours:

In [ ]:
gap_l = np.abs(filtered_l[0, 84:] - filtered_l[1, :84]).max()
gap_t = np.abs(filtered_t[0, 84:] - filtered_t[1, :84]).max()
print(f"window 0 vs 1 on their 84 shared hours, largest difference: load {gap_l:.3g}, temperature {gap_t:.3g} °C")
print(f"before the filter: {max(np.abs(xl[0, 84:] - xl[1, :84]).max(), np.abs(xt[0, 84:] - xt[1, :84]).max()):.3g}")

## 5. `add_sinusoid`: a wave at a known frequency

**On the temperature selection**, a 6-hour oscillation (`frequency: 1/6`) of `amplitude: 1` °C, as in the example config (which writes `0.1666667`). The output minus the input is exactly the requested wave on every window. 6 hours makes 28 whole periods in a week, so the wave sits on one bin: the spectrum shows a peak of about 1 °C there. The phase is counted from each window's first step. With `stride = 84`, two windows start 84 hours apart: 14 whole periods at 6 hours, so they add the same values on their shared hours. At 0.1 cycles per hour, 8.4 periods, they do not.

In [ ]:
SIX_HOURS = 1 / 6
noisy_t = transform(AddSinusoid, xt, frequency=SIX_HOURS, amplitude=1.0)
wave = np.sin(2 * np.pi * SIX_HOURS * steps)
print("output - input is the wave:", np.allclose(noisy_t - xt, wave[None, :, None]))

for frequency in (SIX_HOURS, 0.1):
    added = transform(AddSinusoid, xt, frequency=frequency, amplitude=1.0) - xt
    print(f"frequency {frequency:.4g}: added values differ on the shared hours by up to {np.abs(added[0, 84:] - added[1, :84]).max():.3g} °C")

show_transform(xt, noisy_t, "add_sinusoid every 6 h, amplitude 1 °C", CHANNEL_6, mark=SIX_HOURS)

**On the load selection**, an 8-hour wave (`frequency: 1/8`, 21 periods in a week) of `amplitude: 0.5`.

In [ ]:
noisy_l = transform(AddSinusoid, xl, frequency=1 / 8, amplitude=0.5)
print("output - input is the wave:", np.allclose(noisy_l - xl, 0.5 * np.sin(2 * np.pi * steps / 8)[None, :, None]))
show_transform(xl, noisy_l, "add_sinusoid every 8 h, amplitude 0.5", CHANNEL_0, mark=1 / 8)

## 6. `jitter` and the seed

`jitter` adds independent Gaussian noise of standard deviation `sigma` to every step of the window. It draws random numbers, so it **requires** a `seed`, and the seed is recorded on every output row. Each window draws from its own generator, seeded by the seed, its id and the parameters, not by its position in a batch. White noise has the same expected amplitude at every frequency: on the spectrum it raises the floor.

**On the load selection**, `sigma = 0.2`.

In [ ]:
jittered_l = transform(Jitter, xl, sigma=0.2, seed=0)
print(f"load noise std {(jittered_l - xl).std():.4f} (sigma 0.2)")
show_transform(xl, jittered_l, "jitter, sigma 0.2", CHANNEL_0)

**On the temperature selection**, `sigma = 0.3` °C, as in the example config: a noisier probe.

What the seed guarantees, on this configuration:

In [ ]:
SIGMA = 0.3
JITTER = {"sigma": SIGMA, "seed": 0}
base = transform(Jitter, xt, **JITTER)
noise = base - xt

print(f"temperature noise mean {noise.mean():+.4f}, std {noise.std():.4f} (sigma {SIGMA})")
print("same seed, same output:     ", np.array_equal(base, transform(Jitter, xt, **JITTER)))
print("other seed, other output:   ", not np.array_equal(base, transform(Jitter, xt, **{**JITTER, "seed": 1})))

# The batch split and the order do not matter: each window keeps its own draw.
one_by_one = np.concatenate([transform(Jitter, xt[i : i + 1], [ids[i]], **JITTER) for i in range(len(xt))])
order = np.random.default_rng(0).permutation(len(xt))
shuffled = transform(Jitter, xt[order], [ids[i] for i in order], **JITTER)
print("window by window = batch:   ", np.array_equal(one_by_one, base))
print("shuffled batch, same draws: ", np.array_equal(shuffled, base[order]))

try:
    Jitter({"sigma": SIGMA})
except ValueError as error:
    print("no seed:", error)

show_transform(xt, base, "jitter, sigma 0.3 °C", CHANNEL_6)

Each window draws its own noise, so two overlapping windows get **different** noise on the hours they share: the hour is one measurement, but it is jittered twice, differently. Section 7 shows how the key avoids that.

The parameters enter the seed too. Two values of `sigma` in a sweep therefore draw **independent** noises, not one noise scaled: the variants of one window are not correlated.

In [ ]:
small = transform(Jitter, xt, **JITTER) - xt
large = transform(Jitter, xt, **{**JITTER, "sigma": 0.6}) - xt
print(f"correlation between the sigma=0.3 and sigma=0.6 noises: {np.corrcoef(small.ravel(), large.ravel())[0, 1]:+.3f}")
print(f"windows 0 and 1, noise on their shared hours differs by up to {np.abs(small[0, 84:] - small[1, :84]).max():.3f} °C")

## 7. Using the key to keep outputs consistent (not coded)

Sections 4 to 6 transform each window on its own. That is the contract today: a transformation receives `(B, W, C)` windows, and the key stays in the selection. It has a cost. An hour shared by two overlapping windows gets two different outputs (`low_pass`, `jitter`, `add_sinusoid` at most frequencies), and two transformers recording the same hour get unrelated perturbations, whereas a real disturbance (a heat wave, a grid event) hits both at once.

The key is enough to fix both. The sketches below compute, **in notebook code**, what a transformation given the key would do. None of this is in the package: it would need the key, and for two of them the whole series, to reach the transformation.

| Relation used | Sketch | Consistency it gives |
|---|---|---|
| overlap (`series_id`, `start_window`) | `jitter` drawn once per series, each window takes its slice | a shared hour gets one noise value |
| simultaneity (`t0_series`, `start_window`) | `add_sinusoid` with its phase taken on the shared clock | one disturbance, seen identically by every series recording at that hour |
| overlap, whole series | `low_pass` on the whole series, then cut into windows | shared hours agree, no ringing at the window edges |

### 7.1 Overlap: one noise per series, not per window

Same configuration as section 6: `sigma = 0.3` °C on the temperature selection. The noise is drawn once for the whole series, from the seed and the series id, and each window takes the slice `[start_window, end_window]`. Two overlapping windows then carry the same noise on their shared hours, and a window's noise still never depends on the batch.

In [ ]:
LENGTHS = {sid: len(v) for sid, v in zip(table.column("series_id").to_pylist(), table.column("values").to_pylist())}
rows_h1 = keys[mask]


def series_noise(series_id, sigma, seed):
    # The noise of a whole series: one draw on its timeline, seeded by the seed and the series id.
    rng = np.random.default_rng([seed, int(hashlib.sha1(series_id.encode()).hexdigest()[:16], 16)])
    return rng.normal(0.0, sigma, size=(LENGTHS[series_id], 1))


def jitter_on_series(xs, rows, sigma, seed):
    noises = {sid: series_noise(sid, sigma, seed) for sid in rows.series_id.unique()}
    return xs + np.stack([noises[r.series_id][r.start_window : r.end_window + 1] for r in rows.itertuples()])


per_window = transform(Jitter, xt, **JITTER)
per_series = jitter_on_series(xt, rows_h1, sigma=SIGMA, seed=0)
for name, out in [("jitter per window", per_window), ("jitter per series", per_series)]:
    noise = out - xt
    gap = np.abs(noise[0, 84:] - noise[1, :84]).max()
    print(f"{name:<18} temperature, windows 0 and 1: noise on shared hours differs by up to {gap:.3f}; std {noise.std():.4f}")

### 7.2 Simultaneity: one disturbance on the shared clock

`add_sinusoid` counts its phase from each window's first step. To model one disturbance seen by both transformers, the phase must be counted on the shared clock instead: for a window, `phase = 2π · frequency · (t0_series + start_window)`. The existing `AddSinusoid` does the rest; the key only sets its `phase`. As in section 5, it goes on the temperature selection. At `frequency = 0.1`, where overlapping windows disagreed in section 5, we take window 1 of `ETTh1` and the first window of `ETTh2`: they share 84 hours of the clock.

- overlapping windows of one series now add the same values on their shared hours;
- simultaneous windows of two series add the same value at the same hour.

In [ ]:
FREQUENCY, AMPLITUDE = 0.1, 1.0


def disturbance_on_clock(xs, rows):
    clock_start = (rows.t0_series + rows.start_window).to_numpy()
    return np.concatenate(
        [
            transform(
                AddSinusoid,
                xs[i : i + 1],
                list(rows.id.iloc[i : i + 1]),
                frequency=FREQUENCY,
                amplitude=AMPLITUDE,
                phase=2 * np.pi * FREQUENCY * c,
            )
            for i, c in enumerate(clock_start)
        ]
    )


pair = keys.iloc[[1, PER_SERIES]]  # ETTh1 window 1 and ETTh2 window 0: simultaneous on 84 hours
xs_pair = data_t[pair.index]
per_window = transform(AddSinusoid, xs_pair, list(pair.id), frequency=FREQUENCY, amplitude=AMPLITUDE) - xs_pair
on_clock = disturbance_on_clock(xs_pair, pair) - xs_pair

a, b = pair.iloc[0], pair.iloc[1]
first = max(a.t0_series + a.start_window, b.t0_series + b.start_window)
last = min(a.t0_series + a.end_window, b.t0_series + b.end_window)
clock = np.arange(first, last + 1)
ia, ib = clock - a.t0_series - a.start_window, clock - b.t0_series - b.start_window
for name, added in [("phase per window", per_window), ("phase on the clock", on_clock)]:
    gap = np.abs(added[0, ia, 0] - added[1, ib, 0]).max()
    print(f"{name:<19} ETTh1 vs ETTh2 at the same hours ({len(clock)}): differ by up to {gap:.3g} °C")

overlap = disturbance_on_clock(xt[:2], rows_h1.iloc[:2]) - xt[:2]
print(f"phase on the clock, ETTh1 windows 0 and 1 on shared hours: differ by up to {np.abs(overlap[0, 84:] - overlap[1, :84]).max():.3g} °C")

fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
for ax, (name, added) in zip(axes, [("phase per window", per_window), ("phase on the clock", on_clock)]):
    for k, row in enumerate([a, b]):
        start = row.t0_series + row.start_window
        ax.plot(np.arange(start, start + len(steps)), added[k, :, 0], lw=1, label=f"{row.series_id}, window from clock hour {start}")
    ax.axvspan(first, last, color="grey", alpha=0.12)
    ax.set_title(f"disturbance added to the temperature, {name}")
axes[0].legend(loc="upper right", fontsize=8)
axes[1].set_xlabel("clock hour (t0_series + step); grey: both windows record")
plt.tight_layout()
plt.show()

### 7.3 Overlap, whole series: filter the series, then cut

`low_pass` needs the whole window, and on a window it rings at both edges (section 4). As there, we filter the temperature at 0.03 cycles per hour. Filtering the whole series once and then cutting the windows removes both problems: overlapping windows read the same filtered hours, and the ringing is left at the two ends of the series only. This needs the transformation to receive the series, not the window: the "series then window" mode, left out of the package for now.

In [ ]:
from kcai_data_sampling_timeseries.transformations import low_pass

whole = low_pass(series[None, :, [6]], CUTOFF_T)[0]  # ETTh1 temperature, filtered once
cut = np.stack([whole[r.start_window : r.end_window + 1] for r in rows_h1.itertuples()])
for name, out in [("low_pass per window", filtered_t), ("low_pass on the series", cut)]:
    gap = np.abs(out[0, 84:] - out[1, :84]).max()
    print(f"{name:<23} windows 0 and 1 differ on shared hours by up to {gap:.3g} °C")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(steps, xt[1, :, 0], lw=0.6, label="before")
ax.plot(steps, filtered_t[1, :, 0], lw=1.2, label="after: low_pass per window")
ax.plot(steps, cut[1, :, 0], lw=1.2, label="after: low_pass on the series, then cut")
ax.set_xlabel("hour in the window")
ax.set_xticks(range(0, 169, 24))
ax.set_ylabel("temperature (°C)")
ax.set_title("temperature selection, window 1")
ax.legend()
plt.show()

## 8. The whole job

The same config, run as the CLI runs it (`kcai-data-sampling process -p examples/config/ett-procedural.yaml`). It writes one ledger row and one `.npy` payload per (window, transformation): 412 windows × 5 transformations, with the `low_pass` sweep expanded into three.

In [ ]:
summary = run({**CFG, "compute": {"progress_bar": False}})
print(summary)

ledger_path = CFG["operations"]["outputs"]["path"].format(selection=LOADER["name"])
ledger = pd.read_parquet(ledger_path)
ledger["seed"] = ledger.seed.astype("Int64")  # NULL for the deterministic rows
print(ledger.columns.tolist())
ledger.groupby(["algorithm", "params"], sort=False).agg(rows=("id", "size"), seed=("seed", "first"))

Each row is enough to replay its output. For a `jitter` row, we take its parent window by `parent_id`, rebuild the transformation from `params` and `seed`, and recompute the payload bit for bit.

In [ ]:
row = ledger[ledger.algorithm == "jitter"].iloc[3]
payloads = Path(CFG["operations"]["outputs"]["samples_dir"].format(selection=LOADER["name"]))
payload = np.load(payloads / row.artifact)

parent = data_t[(keys.id == row.parent_id).to_numpy()]
replayed = transform(Jitter, parent, [row.parent_id], **json.loads(row.params), seed=int(row.seed))
print(row[["id", "parent_id", "series_id", "start_window", "end_window", "seed"]].to_dict())
print(f"noise std of this payload: {(payload - parent[0]).std():.3f} °C")
print("replayed from the row:", np.array_equal(payload, replayed[0]))

## Takeaways

- **The channel is chosen by the data selection, not by the transformation.** Two selections of the same windows, `channels: [0]` (`HUFL`) and `channels: [6]` (`OT`), give one-channel samples with the same keys and ids. Each transformation was applied to each, with values in that channel's units.
- **The key is enough to relate windows.** One-week windows every 84 hours give 206 windows per transformer, 412 in all. Overlap (84 hours), adjacency (lag 0) and lag (84 hours) are computed from `series_id`, `start_window` and `end_window`; the ids are opaque hashes and are never parsed.
- **`t0_series` places the series on one clock, in steps.** Computed from `start`, it is 0 for both transformers: window 0 of `ETTh1` is simultaneous with the first window of `ETTh2` on 168 hours, window 1 on 84, window 2 not at all.
- **Independent windows are reproducible.** The same `seed` draws the same windows, another seed draws others.
- **`low_pass` keeps or removes whole harmonics on a one-week window**, where the daily cycle sits on one bin: at 0.03 cycles per hour it removes the daily cycle of the temperature (1.31 °C on window 0), at 0.06 it keeps the daily cycle of the load and removes its 12-hour harmonic. But real data fills the bins between two cutoffs too, so the 0.03, 0.06 and 0.09 outputs differ by up to 5.09 and 2.88 °C. It is an ideal FFT filter on a short window: two overlapping windows give different outputs on the hours they share (up to 2.13 on the load, 3.37 °C on the temperature).
- **`add_sinusoid` adds exactly the requested wave**, with its phase counted from each window's first step. Every 6 hours, the 84-hour stride holds 14 whole periods and overlapping windows add the same values; at 0.1 cycles per hour they differ by up to 1.9 °C.
- **`jitter` is reproducible and batch-independent.** Its noise has standard deviation 0.3010 for `sigma = 0.3` °C on the temperature, and 0.2008 for `sigma = 0.2` on the load. The same seed gives the same output, another seed another one; window by window, as one batch or shuffled, each window gets the same draw. Without a seed it is refused. Two values of `sigma` draw independent noises (correlation +0.003), not one noise scaled.
- **The key is enough to make transformations consistent, but the package does not use it yet (not coded).** Today each window is transformed alone: overlapping windows get different values on shared hours, and simultaneous transformers get unrelated perturbations. In notebook code, the key fixes both: drawing `jitter` once per series brings the gap on shared hours from 1.326 °C to 0; taking the `add_sinusoid` phase on the shared clock makes both transformers get the same disturbance (gap 1.9 °C → 4e-15); filtering the temperature series before cutting removes the 3.37 °C gap on shared hours. It needs the key, and for `jitter` and `low_pass` the whole series, to reach the transformation.
- **The ledger replays each output.** The job writes 2,060 rows (412 windows × 5 transformations). `seed` is 0 on the `jitter` rows and NULL on the deterministic ones, and a `jitter` payload is recomputed bit for bit from its row and its parent window.